# ตรวจจับไฟจากภาพถ่ายทางอากาศด้วย YOLO26 — เทรนและใช้งานครบในไฟล์เดียว

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🇹🇭 **ภาษาไทย** (เอกสารฉบับนี้) · [🇬🇧 English](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb)

เทรนโมเดลตรวจจับไฟบนภาพถ่ายทางอากาศจากอากาศยานไร้คนขับด้วย
[Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/) แล้วนำไปใช้งานจริงกับทั้งภาพนิ่งและวิดีโอ

โน้ตบุ๊กนี้รวมทั้งสามส่วนไว้ในไฟล์เดียว **ให้รันจากบนลงล่างตามลำดับ**
ส่วนที่ 2 และ 3 จะหยิบไฟล์ `best.pt` ที่เพิ่งเทรนเสร็จในส่วนที่ 1 มาใช้ต่อให้เองอัตโนมัติ
จึงไม่ต้องมานั่งอัปโหลดไฟล์เองเหมือนตอนที่ยังแยกเป็นสามไฟล์

| ส่วน | เนื้อหา | GPU |
|---|---|---|
| [ส่วนที่ 1](#part-1) | เทรนโมเดล YOLO26 กับชุดข้อมูลจาก Roboflow | จำเป็น |
| [ส่วนที่ 2](#part-2) | ตรวจจับไฟบนภาพนิ่ง แล้ววาดผลลัพธ์ด้วย Supervision | ไม่จำเป็น |
| [ส่วนที่ 3](#part-3) | ตรวจจับและติดตามไฟบนวิดีโอด้วย ByteTrack | จำเป็น |

**Runtime:** `Runtime` → `Change runtime type` → **T4 GPU** (หรือดีกว่านั้น) แล้วกด `Save`

<a id="part-1"></a>
<hr>

<h1 align="center">🔥 ส่วนที่ 1 · เทรนโมเดล</h1>

<p align="center">
  <b>Ultralytics YOLO26 + Roboflow</b> — ปรับละเอียด (fine-tune) โมเดลตรวจจับไฟจากชุดข้อมูลภาพถ่ายทางอากาศ<br>
  <i>ส่วนนี้ต้องใช้ GPU และเป็นขั้นตอนที่กินเวลามากที่สุด</i>
</p>

<hr>

## 1. ตรวจสอบ GPU

In [ ]:
!nvidia-smi

## 2. ติดตั้งไลบรารี

กำหนดเป็นเวอร์ชันขั้นต่ำ ไม่ได้ตรึงเวอร์ชันแบบเป๊ะ ๆ เพราะแค่ต้องการการันตีว่า API
ที่โน้ตบุ๊กนี้เรียกใช้มีอยู่จริง แล้วปล่อยให้ pip ไปจับคู่กับ PyTorch รุ่นที่ Colab ติดตั้งมาให้เอง

ติดตั้งให้ครบทั้งสามส่วนตั้งแต่ตรงนี้ทีเดียว จะได้ไม่ต้องกลับมารันซ้ำระหว่างทาง โดยเฉพาะ
`lap` ที่ตัวติดตามวัตถุ ByteTrack ในส่วนที่ 3 ต้องใช้ ซึ่ง Ultralytics ไม่ได้ลงมาให้ตั้งแต่แรก
ถ้าไม่ลงไว้ก่อน `model.track(...)` จะไปเรียก pip ติดตั้งเองกลางลูปประมวลผลวิดีโอ

In [ ]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0" "lap>=0.5.12" "roboflow>=1.4.1"

import supervision as sv
import ultralytics

ultralytics.checks()
print("supervision:", sv.__version__)

In [ ]:
import os
from pathlib import Path

from ultralytics import YOLO

HOME = Path.cwd()
print("HOME:", HOME)

## 3. ดาวน์โหลดชุดข้อมูลจาก Roboflow

ต้องใช้ Roboflow API key แบบไม่มีค่าใช้จ่าย (<https://app.roboflow.com/settings/api>)

ถ้ารันบน Colab ให้เก็บ key ไว้ครั้งเดียวในแผง 🔑 **Secrets** ทางแถบซ้ายมือ ตั้งชื่อว่า
`ROBOFLOW_API_KEY` แล้วเปิดสิทธิ์ *Notebook access* เซลล์ด้านล่างจะอ่านจากที่นั่นก่อน
ถ้าไม่เจอจึงไปอ่านจากตัวแปรสภาพแวดล้อม แล้วค่อยถามผู้ใช้เป็นทางเลือกสุดท้าย
วิธีนี้ทำให้ key ไม่ติดไปกับตัวโน้ตบุ๊กตอน commit

In [ ]:
import getpass
import os


def get_roboflow_api_key() -> str:
    """อ่าน API key จาก Colab Secrets ก่อน ถ้าไม่เจอจึงไปอ่านจากตัวแปรสภาพแวดล้อม แล้วค่อยถามผู้ใช้"""
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("ROBOFLOW_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key
    return getpass.getpass("Roboflow API key: ")


ROBOFLOW_API_KEY = get_roboflow_api_key()

In [ ]:
from roboflow import Roboflow

DATASETS_DIR = HOME / "datasets"
DATASETS_DIR.mkdir(exist_ok=True)
os.chdir(DATASETS_DIR)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tim-4ijf0").project("drone-fire-detection-byija")

# "yolov8" คือชื่อ *รูปแบบการจัดวางไฟล์ตอนส่งออก* ของชุดข้อมูล (ภาพ + ไฟล์ label .txt ตามมาตรฐาน
# YOLO + data.yaml) ไม่ใช่เวอร์ชันของโมเดล ฝั่ง Roboflow ไม่มีรูปแบบส่งออกชื่อ "yolo26" และไม่จำเป็นต้องมี
# เพราะนี่คือรูปแบบ YOLO มาตรฐานที่ YOLO26 เอาไปเทรนต่อได้เลยโดยไม่ต้องแก้อะไร ส่วน data.yaml
# ที่ส่งออกมาจะใช้พาธแบบสัมพัทธ์ "../train/images" ซึ่ง Ultralytics จะไล่หาโดยอิงจากโฟลเดอร์ของ
# data.yaml เอง เราจึงไม่ต้องไปยุ่งกับพาธเพิ่ม
dataset = project.version(1).download("yolov8")

os.chdir(HOME)

DATA_YAML = Path(dataset.location) / "data.yaml"
print("data.yaml:", DATA_YAML)
print(DATA_YAML.read_text())

## 4. เทรนโมเดล

YOLO26 เป็นสถาปัตยกรรมแบบครบวงจร (end-to-end) และไม่ผ่าน NMS จึงไม่มีค่าขีดแบ่ง `iou`
ของ NMS ให้ต้องปรับ เพราะโมเดลคายกรอบสุดท้ายออกมาให้เลย

โค้ดเก็บค่า `results.save_dir` ไว้ เพื่อให้เซลล์ถัด ๆ ไปไม่ต้อง hardcode พาธ `runs/detect/train`
เนื่องจาก Ultralytics จะไล่เลขเป็น `train2`, `train3`, … ทุกครั้งที่รันซ้ำ

ถ้าเจอปัญหาหน่วยความจำไม่พอ (out of memory) ให้ลด `imgsz` เหลือ 640 หรือเปลี่ยน `model` เป็น `yolo26n.pt`

In [ ]:
MODEL_ARCH = "yolo26m.pt"  # n / s / m / l / x

model = YOLO(MODEL_ARCH)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=800,
    plots=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("run dir:", RUN_DIR)
print("best weights:", BEST_WEIGHTS)

In [ ]:
from IPython.display import Image, display

for artifact in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    path = RUN_DIR / artifact
    if path.exists():
        print(artifact)
        display(Image(filename=str(path), width=700))
    else:
        print("ไม่พบไฟล์:", artifact)

## 5. ตรวจสอบความแม่นยำของโมเดล (validate)

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(DATA_YAML))

print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP75:    {metrics.box.map75:.4f}")

## 6. ลองทำนายผลบนชุดทดสอบ (test split)

In [ ]:
predict_results = best_model.predict(
    source=str(Path(dataset.location) / "test" / "images"),
    conf=0.25,
    save=True,
)

PREDICT_DIR = Path(predict_results[0].save_dir)
print("predictions:", PREDICT_DIR)

In [ ]:
import glob

for image_path in sorted(glob.glob(f"{PREDICT_DIR}/*.jpg"))[:3]:
    display(Image(filename=image_path, width=700))

## 7. ส่งออกไฟล์ weights

ส่วนที่ 2 และ 3 ในโน้ตบุ๊กนี้จะหยิบ `best.pt` จากตัวแปร `BEST_WEIGHTS` ไปใช้ต่อให้เอง
เซลล์นี้จึงเป็นแค่ทางเลือก สำหรับตอนที่อยากดาวน์โหลด checkpoint เก็บไว้ใช้ภายนอก

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(BEST_WEIGHTS))
except ImportError:
    print("ไม่ได้รันบน Colab ไฟล์ weights อยู่ที่:", BEST_WEIGHTS)

<a id="part-2"></a>
<hr>

<h1 align="center">🖼️ ส่วนที่ 2 · ตรวจจับไฟบนภาพนิ่ง</h1>

<p align="center">
  <b>YOLO26 + Supervision</b> — รันโมเดลกับภาพหนึ่งภาพ แล้ววาดกรอบและป้ายกำกับ<br>
  <i>ส่วนนี้ไม่ต้องใช้ GPU เพราะ inference กับภาพเดียวรันบน CPU ก็ทันใจ</i>
</p>

<hr>

## 8. เตรียมไฟล์ weights และภาพทดสอบ

ถ้ารันส่วนที่ 1 มาแล้วในเซสชันเดียวกัน เซลล์นี้จะหยิบ `best.pt` ที่เพิ่งเทรนเสร็จมาใช้ทันที
ถ้าเปิดโน้ตบุ๊กขึ้นมาใหม่แล้วข้ามมาที่ส่วนนี้เลย มันจะมองหา `best.pt` ในโฟลเดอร์ปัจจุบันก่อน
แล้วค่อยขอให้อัปโหลดเป็นทางเลือกสุดท้าย ส่วนภาพตัวอย่างจะดึงมาจากคลังโค้ดให้เองอัตโนมัติ

In [ ]:
from pathlib import Path


def resolve_best_weights() -> Path:
    """หา best.pt ตามลำดับ: ของที่เพิ่งเทรนในเซสชันนี้ → ไฟล์ในโฟลเดอร์ปัจจุบัน → ให้ผู้ใช้อัปโหลด"""
    trained = globals().get("BEST_WEIGHTS")
    if trained is not None and Path(trained).exists():
        return Path(trained)

    local = Path("best.pt")
    if local.exists():
        return local

    print("ไม่พบ best.pt — อัปโหลดไฟล์ตอนนี้ได้เลย (หรือย้อนกลับไปรันส่วนที่ 1 ก่อน)")
    try:
        from google.colab import files  # type: ignore

        files.upload()
    except ImportError:
        raise FileNotFoundError("วาง best.pt ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊กนี้")
    if not local.exists():
        raise FileNotFoundError("อัปโหลดแล้วแต่ยังไม่พบ best.pt")
    return local


MODEL_PATH = resolve_best_weights()

IMAGE_PATH = Path("fire_image.png")
if not IMAGE_PATH.exists():
    !wget -q -O {IMAGE_PATH} https://raw.githubusercontent.com/jakkzz/Fire-Detection-Drone/main/fire_image.png

print("model:", MODEL_PATH.resolve())
print("image:", IMAGE_PATH.resolve())

## 9. รัน inference บนภาพนิ่ง

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f"อ่านไฟล์ {IMAGE_PATH} ไม่ได้")

result = model(image, conf=0.25, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

print("detections:", len(detections))

## 10. วาดกรอบและป้ายกำกับ

Supervision รุ่นปัจจุบันแยกการวาดออกเป็นหลาย annotator แล้ว `BoxAnnotator` ไม่รับ
อาร์กิวเมนต์ `labels=` อีกต่อไป (ถูกถอดออกตั้งแต่ supervision 0.22) ป้ายกำกับจึงเป็นหน้าที่ของ
`LabelAnnotator`

ชื่อชั้นข้อมูลอ่านมาจาก `detections["class_name"]` ซึ่ง `Detections.from_ultralytics`
เติมให้จากตัวโมเดลเอง จึงไม่ต้องมานั่งดูแลรายชื่อคลาสเองให้เสี่ยงหลุดไม่ตรงกับ weights ที่ใช้อยู่

In [ ]:
box_annotator = sv.BoxAnnotator(thickness=3)
label_annotator = sv.LabelAnnotator(text_scale=0.8, text_thickness=2, text_padding=6)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections["class_name"], detections.confidence)
]

annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

# OpenCV อ่านภาพมาเป็น BGR ต้องแปลงก่อน สีใน matplotlib จึงจะถูกต้อง
sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))

## 11. บันทึกภาพผลลัพธ์

In [ ]:
OUTPUT_PATH = Path("fire_image_annotated.png")
cv2.imwrite(str(OUTPUT_PATH), annotated)
print("saved:", OUTPUT_PATH.resolve())

<a id="part-3"></a>
<hr>

<h1 align="center">🎥 ส่วนที่ 3 · ตรวจจับและติดตามไฟบนวิดีโอ</h1>

<p align="center">
  <b>YOLO26 + ByteTrack + Supervision</b> — ไล่ทั้งวิดีโอ ติดตามไฟแต่ละจุดข้ามเฟรม แล้วเขียนวิดีโอผลลัพธ์ออกมา<br>
  <i>ส่วนนี้ควรใช้ GPU เพราะ inference วิดีโอบน CPU ช้ามาก</i>
</p>

<hr>

> **สิ่งที่เปลี่ยนไป** เมื่อก่อนการติดตามวัตถุต้องโคลน
> [ByteTrack](https://github.com/ifzhang/ByteTrack) มา build YOLOX จากซอร์ส แล้วติดตั้ง
> `onemetric` กับ `cython_bbox` ซึ่งเป็นชุดเครื่องมือที่ build บน Python รุ่นปัจจุบันไม่ผ่านแล้ว
> ทุกวันนี้ ByteTrack มากับ Ultralytics อยู่แล้ว `model.track(...)` จึงจัดการให้ครบโดยไม่ต้องลง
> อะไรเพิ่ม และ Supervision ก็อ่านหมายเลข track ออกมาจากผลลัพธ์ได้ตรง ๆ

## 12. เตรียมวิดีโอต้นทาง

ใช้ `best.pt` ตัวเดียวกับส่วนที่ 2 ส่วนวิดีโอตัวอย่าง `fire.mp4` จะดึงมาจากคลังโค้ดให้เองอัตโนมัติ

In [ ]:
SOURCE_VIDEO_PATH = Path("fire.mp4")
TARGET_VIDEO_PATH = Path("fire_result.mp4")

if not SOURCE_VIDEO_PATH.exists():
    !wget -q -O {SOURCE_VIDEO_PATH} https://github.com/jakkzz/Fire-Detection-Drone/raw/main/fire.mp4

MODEL_PATH = resolve_best_weights()

video_info = sv.VideoInfo.from_video_path(str(SOURCE_VIDEO_PATH))
print("model:", MODEL_PATH.resolve())
print(video_info)

## 13. โหลดโมเดลสำหรับงานวิดีโอ

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
model.fuse()

print("classes:", model.names)

## 14. ลองวาดผลลัพธ์บนเฟรมเดียวก่อน

เป็นการเช็กความเรียบร้อยแบบถูก ๆ ก่อนจะลงทุนรันยาวทั้งวิดีโอ

In [ ]:
import cv2

CONFIDENCE_THRESHOLD = 0.25

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.6, text_thickness=2, text_padding=5)


def make_labels(detections: sv.Detections) -> list[str]:
    """สร้างป้ายกำกับรูปแบบ `#id class conf` โดยตัดหมายเลข id ออกเมื่อยังไม่ได้เปิดการติดตามวัตถุ"""
    labels = []
    for i in range(len(detections)):
        name = detections["class_name"][i]
        conf = detections.confidence[i]
        tracker_id = None if detections.tracker_id is None else detections.tracker_id[i]
        prefix = "" if tracker_id is None else f"#{tracker_id} "
        labels.append(f"{prefix}{name} {conf:.2f}")
    return labels


frame = next(sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH)))

result = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=make_labels(detections))

sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))

## 15. ตรวจจับ + ติดตามวัตถุตลอดทั้งวิดีโอ

`model.track(..., persist=True, tracker="bytetrack.yaml")` จะเก็บสถานะของตัวติดตามไว้ระหว่าง
การเรียกแต่ละครั้ง ไฟแต่ละจุดจึงได้หมายเลข id ที่คงที่ข้ามเฟรม ส่วน
`sv.Detections.from_ultralytics` จะอ่านหมายเลขเหล่านั้นเข้ามาไว้ใน `detections.tracker_id`
แล้ว `TraceAnnotator` ก็เอาไปวาดเป็นเส้นร่องรอยการเคลื่อนที่

ByteTrack ถูกออกแบบมาให้กินผลตรวจจับที่ค่าความเชื่อมั่น *ต่ำ* เพราะมันดึงกรอบที่มั่นใจน้อย
กลับมาจับคู่กับ track ที่มีอยู่แล้ว ซึ่งเป็นที่มาของความแม่นยำส่วนใหญ่ของมัน โค้ดจึงส่งค่า `conf`
ต่ำ ๆ ให้ `track()` แล้วค่อยไปกรองค่าความเชื่อมั่นที่ผลลัพธ์ทีหลัง แทนที่จะไปอดอาหารตัวติดตาม
ตั้งแต่ขาเข้า

In [ ]:
from tqdm.auto import tqdm

trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=30)

TRACKER_INPUT_CONF = 0.1  # ตั้งต่ำโดยตั้งใจ: ByteTrack เอากรอบที่มั่นใจน้อยไปจับคู่กับ track ที่ยังวิ่งอยู่

frame_generator = sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH))

with sv.VideoSink(str(TARGET_VIDEO_PATH), video_info) as sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):
        result = model.track(
            frame,
            conf=TRACKER_INPUT_CONF,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False,
        )[0]
        detections = sv.Detections.from_ultralytics(result)
        # ติดตามจากกรอบที่มั่นใจน้อยด้วย แต่วาดเฉพาะกรอบที่มั่นใจพอ
        detections = detections[detections.confidence >= CONFIDENCE_THRESHOLD]

        annotated = frame.copy()
        if detections.tracker_id is not None:
            annotated = trace_annotator.annotate(scene=annotated, detections=detections)
        annotated = box_annotator.annotate(scene=annotated, detections=detections)
        annotated = label_annotator.annotate(
            scene=annotated, detections=detections, labels=make_labels(detections)
        )

        sink.write_frame(annotated)

print("wrote:", TARGET_VIDEO_PATH.resolve())

## 16. เล่นวิดีโอผลลัพธ์ในหน้าโน้ตบุ๊ก

Supervision เขียนไฟล์ออกมาเป็น `mp4v` ซึ่งเบราว์เซอร์ถอดรหัสไม่ได้ ต้อง re-encode เป็น H.264
ด้วย ffmpeg (Colab ติดตั้งมาให้อยู่แล้ว) วิดีโอจึงจะเล่นในหน้าโน้ตบุ๊กได้

In [ ]:
PLAYABLE_VIDEO_PATH = Path("fire_result_h264.mp4")

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -pix_fmt yuv420p {PLAYABLE_VIDEO_PATH}

print("wrote:", PLAYABLE_VIDEO_PATH.resolve())

In [ ]:
import base64

from IPython.display import HTML, display

encoded = base64.b64encode(PLAYABLE_VIDEO_PATH.read_bytes()).decode()
display(HTML(f'''
<video width="720" controls>
  <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
</video>
'''))

## 17. ดาวน์โหลดผลลัพธ์

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(PLAYABLE_VIDEO_PATH))
except ImportError:
    print("ไม่ได้รันบน Colab ไฟล์ผลลัพธ์อยู่ที่:", PLAYABLE_VIDEO_PATH.resolve())